# 05 - GRPO with a local drug-target verifier

This notebook uses a local copy of `drug_bank.csv` supplied by the user at runtime. The source data is not stored in this public repository.

Pipeline: local drug-target table -> compact dictionary -> prompt-only GRPO -> verifiable reward -> drug-disjoint evaluation -> before/after comparison.

**Important:** use the source data only according to its original terms.


In [1]:
!pip -q install -U \
    "transformers>=4.55,<5" \
    "datasets>=3.6,<5" \
    "peft>=0.17,<1" \
    "trl>=0.29,<1" \
    "accelerate>=1.10,<2" \
    "bitsandbytes>=0.46,<1" \
    "torchao>=0.16,<1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 134.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 47.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import ast
import re
import pandas as pd
import torch
from datasets import Dataset

import transformers, datasets, peft, trl
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before running this notebook.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Datasets: 4.8.5
PEFT: 0.20.0
TRL: 0.29.1
CUDA available: True
GPU: NVIDIA L4
VRAM (GB): 22.0


## 1. Upload `drug_bank.csv`

Upload the local source file directly into the Colab runtime. Nothing is fetched from the GitHub repository.


In [3]:
from google.colab import files

uploaded = files.upload()
if "drug_bank.csv" not in uploaded:
    raise FileNotFoundError("Please upload a file named drug_bank.csv")
DATA_PATH = "/content/drug_bank.csv"
print("Using:", DATA_PATH)


Saving drug_bank.csv to drug_bank.csv
Using: /content/drug_bank.csv


In [4]:
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()


Rows: 7756
Columns: ['Unnamed: 0', 'Drug Name', 'DrugBank ID', 'Targets Name', 'Targets', 'SMILES', 'PubChem Compound ID', 'PubChem Substance ID']


,Unnamed: 0,Drug Name,DrugBank ID,Targets Name,Targets,SMILES,PubChem Compound ID,PubChem Substance ID
0,0,Lepirudin,DB00001,['Prothrombin'],['P00734'],NaN,NaN,46507011.0
1,1,Cetuximab,DB00002,"['Epidermal growth factor receptor', 'Low affi...","['P00533', 'O75015', 'P02745', 'P02746', 'P027...",NaN,NaN,46507042.0
2,2,Denileukin diftitox,DB00004,"['Interleukin-2 receptor subunit alpha', 'Inte...","['P01589', 'P14784', 'P31785']",NaN,NaN,46506950.0
3,3,Etanercept,DB00005,"['Tumor necrosis factor', 'Lymphotoxin-alpha',...","['P01375', 'P01374', 'P12314', 'P12318', 'P319...",NaN,NaN,46506732.0
4,4,Bivalirudin,DB00006,['Prothrombin'],['P00734'],CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,16129704.0,46507415.0


## 2. Build a compact drug -> target dictionary

For the first GRPO experiment, keep only drugs with exactly one target in the source table. The gold target is the UniProt ID.


In [5]:
drug_targets = {}
for _, row in df.iterrows():
    drug = str(row["Drug Name"]).strip()
    try:
        target_ids = ast.literal_eval(str(row["Targets"]))
    except Exception:
        target_ids = []
    try:
        target_names = ast.literal_eval(str(row["Targets Name"]))
    except Exception:
        target_names = []
    pairs = list(zip(target_ids, target_names))
    if drug and pairs:
        drug_targets[drug] = pairs

single_target = {drug: pairs[0] for drug, pairs in drug_targets.items() if len(pairs) == 1}
print("Drugs with exactly one target:", len(single_target))
print(list(single_target.items())[:10])


Drugs with exactly one target: 4909
[('Lepirudin', ('P00734', 'Prothrombin')), ('Bivalirudin', ('P00734', 'Prothrombin')), ('Leuprolide', ('P30968', 'Gonadotropin-releasing hormone receptor')), ('Sermorelin', ('Q02643', 'Growth hormone-releasing hormone receptor')), ('Darbepoetin alfa', ('P19235', 'Erythropoietin receptor')), ('Erythropoietin', ('P19235', 'Erythropoietin receptor')), ('Salmon calcitonin', ('P30988', 'Calcitonin receptor')), ('Pegfilgrastim', ('Q99062', 'Granulocyte colony-stimulating factor receptor')), ('Thyrotropin alfa', ('P16473', 'Thyrotropin receptor')), ('Anakinra', ('P14778', 'Interleukin-1 receptor type 1'))]


## 3. Build a prompt-only GRPO dataset

GRPO receives prompts. The reward function can access extra dataset columns such as the gold target. We split by drug so evaluation uses unseen drugs.


In [6]:
records = []
for drug, (target_id, target_name) in single_target.items():
    records.append({
        "prompt": [{
            "role": "user",
            "content": f"What is the protein target UniProt ID for the drug {drug}? Reply with the UniProt ID only."
        }],
        "drug": drug,
        "target_id": target_id,
        "target_name": target_name,
    })

dataset = Dataset.from_list(records).shuffle(seed=42)
split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print("Total:", len(dataset))
print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print(train_dataset[0])


Total: 4909
Train: 3927
Eval: 982
{'prompt': [{'role': 'user', 'content': 'What is the protein target UniProt ID for the drug Quinonoid 7,8-Tetrahydrobiopterin? Reply with the UniProt ID only.'}], 'drug': 'Quinonoid 7,8-Tetrahydrobiopterin', 'target_id': 'P00439', 'target_name': 'Phenylalanine-4-hydroxylase'}


## 4. Load Qwen2.5-0.5B-Instruct


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).cuda()
print("Has chat template:", tokenizer.chat_template is not None)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Has chat template: True


## 5. Generation and exact-match evaluation


In [8]:
def make_inputs(drug):
    messages = [{
        "role": "user",
        "content": f"What is the protein target UniProt ID for the drug {drug}? Reply with the UniProt ID only."
    }]
    return tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True)

def generate_answer(model, drug, max_new_tokens=16, do_sample=False):
    encoded = make_inputs(drug)
    encoded = {key: value.to(model.device) for key, value in encoded.items()}
    with torch.no_grad():
        output = model.generate(**encoded, max_new_tokens=max_new_tokens, do_sample=do_sample, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(output[0], skip_special_tokens=True)

def extract_target(text):
    found = re.findall(r"\b[A-Z][0-9][A-Z0-9]{3}[0-9]\b", text.upper())
    return found[0] if found else None

def exact_accuracy(model, dataset):
    rows = []
    correct = 0
    for example in dataset:
        text = generate_answer(model, example["drug"])
        pred = extract_target(text)
        gold = str(example["target_id"]).upper()
        ok = pred == gold
        correct += int(ok)
        rows.append({"drug": example["drug"], "gold": gold, "pred": pred, "correct": ok})
    accuracy = correct / len(dataset) if len(dataset) else float("nan")
    return accuracy, rows

before_acc, before_rows = exact_accuracy(model, eval_dataset)
print(f"Before GRPO exact target accuracy: {before_acc:.3f}")
for row in before_rows[:10]:
    print(row)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Before GRPO exact target accuracy: 0.000
{'drug': '2-(cyclohexylamino)benzoic acid', 'gold': 'O14757', 'pred': 'P01345', 'correct': False}
{'drug': 'Avotaciclib', 'gold': 'P06493', 'pred': 'P01243', 'correct': False}
{'drug': '9,10-Deepithio-9,10-Didehydroacanthifolicin', 'gold': 'P36873', 'pred': 'P32465', 'correct': False}
{'drug': 'SB-409513', 'gold': 'P49841', 'pred': 'P27683', 'correct': False}
{'drug': 'Galidesivir', 'gold': 'Q05318', 'pred': 'P01243', 'correct': False}
{'drug': "(2Z)-2-cyano-N-(3'-ethoxybiphenyl-4-yl)-3-hydroxybut-2-enamide", 'gold': 'Q02127', 'pred': 'P01658', 'correct': False}
{'drug': '5-Bromo-N-[(2S)-2,3-dihydroxypropoxy]-3,4-difluoro-2-[(2-fluoro-4-iodophenyl)amino]benzamide', 'gold': 'Q02750', 'pred': 'P01678', 'correct': False}
{'drug': '(2S)-1-[(2S,4R)-4-Benzyl-2-hydroxy-5-{[(1S,2R,5S)-2-hydroxy-5-methylcyclopentyl]amino}-5-oxopentyl]-4-{[6-chloro-5-(4-methyl-1-piperazinyl)-2-pyrazinyl]carbonyl}-N-(2-methyl-2-propanyl)-2-piperazineca rboxamide', 'gold': 

In [13]:
def exact_accuracy_batched(model, dataset, batch_size=16):
    model.eval()

    correct = 0
    rows = []

    for start in range(0, len(dataset), batch_size):
        batch = dataset[start:start + batch_size]

        messages = [
            [
                {
                    "role": "user",
                    "content": (
                        f"What is the protein target UniProt ID for the drug "
                        f"{drug}? Reply with the UniProt ID only."
                    ),
                }
            ]
            for drug in batch["drug"]
        ]

        prompts = [
            tokenizer.apply_chat_template(
                msg,
                tokenize=False,
                add_generation_prompt=True,
            )
            for msg in messages
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=8,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        # Remove the padded prompt tokens before decoding.
        input_length = inputs["input_ids"].shape[1]
        generated = outputs[:, input_length:]

        texts = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True,
        )

        for drug, gold, text in zip(
            batch["drug"],
            batch["target_id"],
            texts,
        ):
            pred = extract_target(text)
            ok = pred == gold.upper()

            correct += int(ok)

            rows.append(
                {
                    "drug": drug,
                    "gold": gold,
                    "pred": pred,
                    "correct": ok,
                    "text": text,
                }
            )

        print(
            f"Processed {min(start + batch_size, len(dataset))}"
            f"/{len(dataset)}"
        )

    accuracy = correct / len(dataset) if len(dataset) else float("nan")

    return accuracy, rows

## 6. Verifiable reward

The reward is 1 when the generated answer contains the gold UniProt ID, otherwise 0. This is intentionally simple and fully deterministic.


In [9]:
def target_reward(completions, target_id, **kwargs):
    rewards = []
    for completion, gold in zip(completions, target_id):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        found = re.findall(r"\b[A-Z][0-9][A-Z0-9]{3}[0-9]\b", text.upper())
        rewards.append(1.0 if str(gold).upper() in found else 0.0)
    return rewards


## 7. GRPO + LoRA

GRPO samples multiple candidate completions and uses the verifier reward to update the policy.


In [11]:
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

grp_args = GRPOConfig(
    output_dir="./outputs/drug-target-grpo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_steps=100,
    num_generations=4,
    max_completion_length=16,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    use_vllm=False,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)

trainer = GRPOTrainer(
    model=model,
    args=grp_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    reward_funcs=target_reward,
    peft_config=lora_config,
)
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000


TrainOutput(global_step=100, training_loss=0.0, metrics={'train_runtime': 190.0279, 'train_samples_per_second': 2.105, 'train_steps_per_second': 0.526, 'total_flos': 0.0, 'train_loss': 0.0})

## 8. After-GRPO evaluation


In [14]:
after_acc, after_rows = exact_accuracy_batched(
    model,
    eval_dataset,
    batch_size=16,
)

print(f"After GRPO exact target accuracy: {after_acc:.3f}")

Processed 16/982
Processed 32/982
Processed 48/982
Processed 64/982
Processed 80/982
Processed 96/982
Processed 112/982
Processed 128/982
Processed 144/982
Processed 160/982
Processed 176/982
Processed 192/982
Processed 208/982
Processed 224/982
Processed 240/982
Processed 256/982
Processed 272/982
Processed 288/982
Processed 304/982
Processed 320/982
Processed 336/982
Processed 352/982
Processed 368/982
Processed 384/982
Processed 400/982
Processed 416/982
Processed 432/982
Processed 448/982
Processed 464/982
Processed 480/982
Processed 496/982
Processed 512/982
Processed 528/982
Processed 544/982
Processed 560/982
Processed 576/982
Processed 592/982
Processed 608/982
Processed 624/982
Processed 640/982
Processed 656/982
Processed 672/982
Processed 688/982
Processed 704/982
Processed 720/982
Processed 736/982
Processed 752/982
Processed 768/982
Processed 784/982
Processed 800/982
Processed 816/982
Processed 832/982
Processed 848/982
Processed 864/982
Processed 880/982
Processed 896/98

## 9. Before vs After

The same drug-disjoint evaluation set is used for both measurements.


In [15]:
print("Metric comparison")
print("-" * 70)
print(f"{'Metric':<30}{'Before':>12}{'After':>12}{'Delta':>12}")
print("-" * 70)
print(f"{'Exact target accuracy':<30}{before_acc:>12.3f}{after_acc:>12.3f}{after_acc-before_acc:>+12.3f}")
print("-" * 70)


Metric comparison
----------------------------------------------------------------------
Metric                              Before       After       Delta
----------------------------------------------------------------------
Exact target accuracy                0.000       0.000      +0.000
----------------------------------------------------------------------


## 10. Inspect a few predictions

Read these outputs critically. Exact matching measures retrieval against the source table; it does not establish clinical correctness.


In [16]:
for before, after in zip(before_rows[:5], after_rows[:5]):
    print("=" * 80)
    print("DRUG:", before["drug"])
    print("GOLD:", before["gold"])
    print("BEFORE PRED:", before["pred"])
    print("AFTER PRED :", after["pred"])


DRUG: 2-(cyclohexylamino)benzoic acid
GOLD: O14757
BEFORE PRED: P01345
AFTER PRED : P01345
DRUG: Avotaciclib
GOLD: P06493
BEFORE PRED: P01243
AFTER PRED : P01243
DRUG: 9,10-Deepithio-9,10-Didehydroacanthifolicin
GOLD: P36873
BEFORE PRED: P32465
AFTER PRED : None
DRUG: SB-409513
GOLD: P49841
BEFORE PRED: P27683
AFTER PRED : None
DRUG: Galidesivir
GOLD: Q05318
BEFORE PRED: P01243
AFTER PRED : P01235


## Next experiments

1. Use all valid targets per drug instead of restricting to single-target drugs.
2. Add a format reward plus a target reward.
3. Replace exact-match-only reward with partial credit or hard-negative-aware reward.
4. Compare GRPO against SFT and DPO on the same drug-disjoint benchmark.
